# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iankitthakur/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [29]:
!pip -q install duckdb
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

frame = con.sql("""
SELECT
  client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS early_impressions,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS early_clicks,
  SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15') * 1.0
    / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15'), 0) AS early_ctr,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS early_avg_position,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS early_active_days,
  SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '2026-03-15') AS late_impressions
FROM march
WHERE gsc_data_available IS TRUE
  AND hash(content_hash_id) % 50 = 0
GROUP BY 1, 2
""").df()

df = frame[frame["early_impressions"] >= 20].copy()
df["late_impressions"] = df["late_impressions"].fillna(0)
# label: used ONLY to check signals and to evaluate, never as an input to the rule
df["label_declining"] = ((df["late_impressions"] / 16) < 0.8 * (df["early_impressions"] / 15)).astype(int)
print(df.shape, "declining rate:", round(df["label_declining"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2191, 9) declining rate: 0.329


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Lane (locked):** which content pages need attention, using early-window Search Console signals only (days 1–15 March). The label (16–31 March) is used only to check signals and to evaluate, never as a rule input.

**Signal 1 (CTR vs position, behind the CTR-fix flag):** Pages that rank closer to the top should get a higher click-through rate, and a page with a much lower CTR than peers at the same position is a candidate for a title/meta rewrite.

In [31]:
d = df.dropna(subset=["early_avg_position", "early_ctr"]).copy()
d["pos_bucket"] = pd.cut(d["early_avg_position"], [0, 3, 10, 20, 100], labels=["1-3", "4-10", "11-20", "21+"])
g = d.groupby("pos_bucket", observed=True)
sig1 = pd.DataFrame({
    "n": g["content_hash_id"].count(),
    "median_ctr_pct": (g["early_ctr"].median() * 100).round(3),
    "pooled_ctr_pct": (g["early_clicks"].sum() / g["early_impressions"].sum() * 100).round(3),
    "declining_rate": g["label_declining"].mean().round(3),
})
sig1

,n,median_ctr_pct,pooled_ctr_pct,declining_rate
pos_bucket,,,,
1-3,219,0.225,0.466,0.311
4-10,993,0.065,0.291,0.354
11-20,440,0.000,0.405,0.323
21+,538,0.000,0.120,0.292


**Verdict: MIXED.** Pooled CTR is highest at positions 1-3 (0.466%, n=219) and lowest at 21+ (0.120%, n=538), and median CTR also falls (0.225%, 0.065%, 0.000%, 0.000%). But pooled CTR at 11-20 (0.405%, n=440) is higher than at 4-10 (0.291%, n=993), so the pattern is not steady; a few high-volume pages may dominate a pooled number. Declining rate by position (0.311, 0.354, 0.323, 0.292) shows no pattern. Position matters for clicks in a rough, directional way, but it does not predict decline by itself.

**Signal 2 (volume, behind the quick-win flag):** Pages with more impressions should be less likely to be declining.

In [32]:
d2 = df.copy()
d2["vol_bucket"] = pd.qcut(d2["early_impressions"], 4, duplicates="drop")
sig2 = d2.groupby("vol_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    declining_rate=("label_declining", "mean")).round(3)
sig2

,n,declining_rate
vol_bucket,,
"(19.999, 83.0]",553,0.335
"(83.0, 278.0]",544,0.316
"(278.0, 997.0]",546,0.310
"(997.0, 86860.0]",548,0.354


**Verdict: MIXED.** Declining rate is 0.335, 0.316, 0.310 and 0.354 across the four volume buckets (about 550 pages each). It moves only a few points with no steady direction, and the highest-volume bucket is the highest, the opposite of my claim. Volume alone is not a useful signal for declining in this sample. This is an observed, directional result.

**Rule idea:** Signal 1 shows position matters for clicks, so I rank pages by how many clicks they lose compared with peers at the same position. Score = (peer CTR for the page's position bucket × its early impressions) − its actual early clicks, never below 0. Peer CTR is computed from days 1–15 only. Reason code: CTR_BELOW_PEERS_AT_POSITION. Action: rewrite title and meta description. No label or future-window column is used in the score.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [34]:
r = df.dropna(subset=["early_avg_position", "early_ctr"]).copy()
r = r[r["early_avg_position"] <= 20].copy()
r["pos_bucket"] = pd.cut(r["early_avg_position"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"]).astype(str)

sums = r.groupby("pos_bucket")[["early_clicks", "early_impressions"]].sum()
peer_ctr = sums["early_clicks"] / sums["early_impressions"]
r["peer_ctr"] = r["pos_bucket"].map(peer_ctr)

r["score"] = (r["peer_ctr"] * r["early_impressions"] - r["early_clicks"]).clip(lower=0).round(1)
r["reason_code"] = "CTR_BELOW_PEERS_AT_POSITION"
r["action"] = "Rewrite title and meta description"

queue = r[r["score"] > 0].sort_values("score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

cols = ["rank", "content_hash_id", "score", "reason_code", "action",
        "early_impressions", "early_clicks", "early_ctr", "early_avg_position", "peer_ctr"]
os.makedirs("work/outputs", exist_ok=True)
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("queue size:", len(queue))
queue[cols].head(10)

queue size: 1171


,rank,content_hash_id,score,reason_code,action,early_impressions,early_clicks,early_ctr,early_avg_position,peer_ctr
0,1,content_7c6373141eae744a,201.4,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,86860.0,51.0,0.000587,5.785512,0.002906
1,2,content_df79670545938b6b,56.2,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,30687.0,33.0,0.001075,6.834084,0.002906
2,3,content_d61fc394d10cba41,53.9,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,18554.0,0.0,0.000000,3.391745,0.002906
3,4,content_3472d0fe6ee11174,35.2,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,22080.0,29.0,0.001313,4.175640,0.002906
4,5,content_3e8fc220d47a1ef9,34.8,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,21283.0,27.0,0.001269,3.109093,0.002906
5,6,content_66d1fffc91f4f029,32.1,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,15096.0,29.0,0.001921,12.638678,0.004049
6,7,content_eca6c694288d7e32,32.0,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,13526.0,31.0,0.002292,2.950083,0.004657
7,8,content_16d8f69719b786ae,30.8,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,8976.0,11.0,0.001225,2.939943,0.004657
8,9,content_d195cb9b8381472b,28.6,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,6572.0,2.0,0.000304,1.966830,0.004657
9,10,content_8028c5009353ea97,28.2,CTR_BELOW_PEERS_AT_POSITION,Rewrite title and meta description,11086.0,4.0,0.000361,3.273758,0.002906


In [35]:
top50 = queue.head(50)
print("Precision@50 (share of top 50 that are declining):", round(top50["label_declining"].mean(), 3))
print("Base rate (all pages):", round(df["label_declining"].mean(), 3))

Precision@50 (share of top 50 that are declining): 0.38
Base rate (all pages): 0.329


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [37]:
review = queue.head(20)[["rank", "score", "early_impressions", "early_clicks", "early_ctr", "peer_ctr", "early_avg_position"]].copy()
review["early_ctr_pct"] = (review["early_ctr"] * 100).round(2)
review["peer_ctr_pct"] = (review["peer_ctr"] * 100).round(2)
review["position"] = review["early_avg_position"].round(1)
review[["rank", "score", "early_impressions", "early_clicks", "early_ctr_pct", "peer_ctr_pct", "position"]]

,rank,score,early_impressions,early_clicks,early_ctr_pct,peer_ctr_pct,position
0,1,201.4,86860.0,51.0,0.06,0.29,5.8
1,2,56.2,30687.0,33.0,0.11,0.29,6.8
2,3,53.9,18554.0,0.0,0.00,0.29,3.4
3,4,35.2,22080.0,29.0,0.13,0.29,4.2
4,5,34.8,21283.0,27.0,0.13,0.29,3.1
5,6,32.1,15096.0,29.0,0.19,0.40,12.6
6,7,32.0,13526.0,31.0,0.23,0.47,3.0
7,8,30.8,8976.0,11.0,0.12,0.47,2.9
8,9,28.6,6572.0,2.0,0.03,0.47,2.0
9,10,28.2,11086.0,4.0,0.04,0.29,3.3


**Rank 1:** Action: rewrite title and meta description. Why: position 5.8, CTR 0.06% vs 0.29% for peers, 86,860 impressions, about 201 clicks below peers (my score). Wrong if: this is one very large page and its score is 3.6 times the next one, so a data quirk on this single page could distort the queue.
**Rank 2:** Action: rewrite title and meta. Why: position 6.8, CTR 0.11% vs 0.29%, 30,687 impressions, about 56 clicks below peers. Wrong if: its average position hides a wide spread across queries, so the true position is worse than 6.8.
**Rank 3:** Action: rewrite title and meta. Why: position 3.4, 0 clicks from 18,554 impressions vs 0.29% for peers, about 54 clicks below peers. Wrong if: clicks are not recorded for this page (a tracking gap) or the impressions come from queries that are answered directly in the results page.
**Rank 4:** Action: rewrite title and meta. Why: position 4.2, CTR 0.13% vs 0.29%, 22,080 impressions, about 35 clicks below peers. Wrong if: the 15-day window is too short and this is a normal dip.
**Rank 5:** Action: rewrite title and meta. Why: position 3.1, CTR 0.13% vs 0.29%, 21,283 impressions, about 35 clicks below peers. Wrong if: ads or a search feature above it take clicks no matter how good the title is.
**Rank 6:** Action: rewrite title and meta. Why: position 12.6, CTR 0.19% vs 0.40%, 15,096 impressions, about 32 clicks below peers. Wrong if: at position 12 clicks depend mostly on page 2 behaviour, so the title is not the problem.
**Rank 7:** Action: rewrite title and meta. Why: position 3.0, CTR 0.23% vs 0.47%, 13,526 impressions, about 32 clicks below peers. Wrong if: the page mostly gets brand-name searches, where CTR naturally differs from other pages.
**Rank 8:** Action: rewrite title and meta. Why: position 2.9, CTR 0.12% vs 0.47%, 8,976 impressions, about 31 clicks below peers. Wrong if: the average position of 2.9 mixes days at position 1 and days off the first page.
**Rank 9:** Action: rewrite title and meta. Why: position 2.0, CTR 0.03% vs 0.47%, 6,572 impressions, about 29 clicks below peers. Wrong if: the page is mostly seen in a format (such as an answer box) where users get what they need without clicking.
**Rank 10:** Action: rewrite title and meta. Why: position 3.3, CTR 0.04% vs 0.29%, 11,086 impressions, about 28 clicks below peers. Wrong if: the early-March pattern is seasonal and returns to normal in the second half of the month.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [39]:
weak = queue.head(50)
weak = weak[weak["label_declining"] == 0][["rank", "score", "early_impressions", "early_ctr", "peer_ctr", "early_avg_position"]].head(5)
weak

,rank,score,early_impressions,early_ctr,peer_ctr,early_avg_position
1,2,56.2,30687.0,0.001075,0.002906,6.834084
2,3,53.9,18554.0,0.000000,0.002906,3.391745
4,5,34.8,21283.0,0.001269,0.002906,3.109093
5,6,32.1,15096.0,0.001921,0.004049,12.638678
6,7,32.0,13526.0,0.002292,0.004657,2.950083


In my top 50, 38% declined in 16–31 March, so 62% did not (base rate for all pages is 32.9%). Precision@50 is 0.38 against a base rate of 0.329, only about 5 points better than picking pages at random, and with 50 pages that gap is within normal noise (about 7 points). The weak picks above (for example ranks 2, 3, 5, 6 and 7) show the rule finds pages that lose clicks relative to peers, which is not the same as pages that are declining over time. My Week-5 model has to beat Precision@50 of 0.38.

Leakage check: the score uses only early_impressions, early_clicks, early_ctr and early_avg_position from 1–15 March, and peer CTR from the same window. The label (16–31 March) is used only after the ranking, to compute Precision@50 and find weak picks. No product flags or future-window columns are inputs.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.